# 06 - Model Training, Validation, Threshold Handoff, and Experiment Tracking

## Canadian Retail Credit Risk Analytics

This is the **final professional Data Scientist version** of Notebook 06.

The modelling focus is deliberately narrow and defensible:

1. **XGBoost** as the main gradient-boosting model.
2. **Random Forest** as the main tree-ensemble challenger.
3. Logistic Regression only as an auditable baseline.
4. Optional train-only over/under-sampling challengers, not as the default strategy.

The goal is not to produce an artificially perfect confusion matrix. The goal is to build a leakage-safe, auditable, realistic credit-risk model that can be defended in a banking model-risk review.

## Professional modelling rules used here

This notebook follows these rules:

| Rule | Decision |
|---|---|
| Split handling | Use the saved `train`, `validation`, and `test` split from Notebook 05 |
| Preprocessing | Fit imputers, encoders, scalers, winsorization, and skewness transformers on training only |
| Sampling | Optional challengers only; applied inside train-only imblearn pipelines |
| Tuning | Random Forest uses train-only CV; XGBoost uses train fit + validation objective |
| Model selection | Compare PR-AUC/ROC-AUC plus business cost and review rate |
| Thresholds | Select thresholds on validation only |
| Test set | Use only once for final confirmation |
| Neptune | Optional; configured through environment variables only |

## Why the older perfect confusion matrix is rejected

A near-perfect confusion matrix in credit default prediction is usually a warning sign, not a success sign.

This notebook avoids the major causes of unrealistic performance:

- No oversampling before train/test split.
- No encoding before train/test split.
- No test-set Hyperopt tuning.
- No test-set threshold selection.
- No leakage-prone repayment variables as model inputs.
- No hard-coded Neptune token.

A credible model with a lower confusion matrix is better than a leaked model with a perfect confusion matrix.

## Neptune configuration

Do **not** paste your Neptune API token inside this notebook.

Windows PowerShell persistent setup:

```powershell
setx NEPTUNE_PROJECT "your-workspace/your-project"
setx NEPTUNE_API_TOKEN "your-token"
setx ENABLE_NEPTUNE "1"
```

Restart VS Code/Jupyter after running `setx`.

Local-only tracking:

```powershell
setx ENABLE_NEPTUNE "0"
```

Fast test run:

```powershell
$env:ENABLE_NEPTUNE="0"
$env:XGB_HYPEROPT_MAX_EVALS="10"
$env:RF_RANDOM_SEARCH_N_ITER="8"
$env:MODEL_CV_FOLDS="3"
```

Full professional run:

```powershell
$env:XGB_HYPEROPT_MAX_EVALS="35"
$env:RF_RANDOM_SEARCH_N_ITER="25"
$env:MODEL_CV_FOLDS="3"
```

## Optional sampling challengers

Sampling is **not** enabled by default because XGBoost already uses `scale_pos_weight` and Random Forest uses class weighting. However, professional modelling should test sampling as a controlled challenger.

To enable train-only sampling challengers:

```powershell
pip install imbalanced-learn
$env:ENABLE_SAMPLING_CHALLENGERS="1"
$env:SAMPLER_STRATEGY="0.50"
```

This adds:

- `random_forest_under_sampled_challenger`
- `random_forest_over_sampled_challenger`
- `xgboost_under_sampled_challenger`
- `xgboost_over_sampled_challenger`

These are fitted only on the training split. Validation and test are never resampled.

In [ ]:
# Imports and project setup

from pathlib import Path
import os
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from credit_risk.config import (
    PROCESSED_DIR,
    TABLE_DIR,
    MODEL_ARTIFACT_DIR,
    ensure_project_directories,
)
from credit_risk.models.experiment_tracking import neptune_config_status
from credit_risk.models.train import TrainingConfig, run_training_workflow_from_file

ensure_project_directories()

In [ ]:
# Configuration review

neptune_status = neptune_config_status()
display(neptune_status)

config = TrainingConfig.from_environment()
config_summary = pd.DataFrame([config.__dict__])
display(config_summary)

# Safety reminder: never print NEPTUNE_API_TOKEN.
print("ENABLE_NEPTUNE:", os.getenv("ENABLE_NEPTUNE", "0"))
print("NEPTUNE_PROJECT set:", bool(os.getenv("NEPTUNE_PROJECT")))
print("Sampling challengers enabled:", config.enable_sampling_challengers)

,setting,status,required_for_neptune
0,ENABLE_NEPTUNE,False,True
1,NEPTUNE_PROJECT,missing,True
2,NEPTUNE_API_TOKEN,missing,True
3,NEPTUNE_MODE,async,False


,random_state,rf_random_search_iter,xgb_hyperopt_max_evals,cv_folds,use_random_under_sampler,false_negative_cost,false_positive_cost,preferred_threshold_objective,n_jobs,enable_sampling_challengers,sampler_strategy,operational_review_rate_cap,operational_min_recall
0,42,25,35,3,False,5000.0,500.0,minimum_cost_review_rate_le_30pct,-1,False,0.5,0.3,0.0


ENABLE_NEPTUNE: 0
NEPTUNE_PROJECT set: False
Sampling challengers enabled: False


In [ ]:
# Load modelling dataset created by Notebook 05

MODEL_INPUT_PATH = PROCESSED_DIR / "credit_risk_modeling_dataset.csv"

if not MODEL_INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Missing modelling dataset: {MODEL_INPUT_PATH}. "
        "Run Notebook 05 or scripts/run_feature_engineering_pipeline.py first."
    )

modeling_df = pd.read_csv(MODEL_INPUT_PATH, low_memory=False)
print("Input shape:", modeling_df.shape)
display(modeling_df["split"].value_counts(dropna=False).rename_axis("split").reset_index(name="rows"))
display(modeling_df.groupby("split")["defaulter"].mean().rename("default_rate").reset_index())

Input shape: (134417, 44)


,split,rows
0,train,94091
1,validation,20163
2,test,20163


,split,default_rate
0,test,0.090413
1,train,0.090412
2,validation,0.090413


## Candidate model design

This notebook fits the following default candidates:

| Candidate | Purpose |
|---|---|
| Logistic Regression | Baseline benchmark only |
| Random Forest weighted baseline | Main RF baseline |
| Random Forest randomized search | Tuned RF challenger |
| XGBoost weighted baseline | Main XGBoost baseline |
| XGBoost Hyperopt tuned | Main tuned XGBoost candidate |
| XGBoost Hyperopt tuned with skew treatment | Tests whether Yeo-Johnson helps numeric features |

Optional sampling challengers can be enabled through environment variables and are treated as challenger experiments, not as automatic winners.

In [ ]:
# Run final Notebook 06 workflow

artifacts = run_training_workflow_from_file(
    modeling_dataset_path=MODEL_INPUT_PATH,
    table_dir=TABLE_DIR,
    model_artifact_dir=MODEL_ARTIFACT_DIR,
    config=config,
)

print("Notebook 06 model training completed.")
print("Ranking champion by validation PR-AUC:", artifacts.ranking_champion_model_name)
print("Operational champion after validation thresholding:", artifacts.operational_champion_model_name)
print("Saved champion model artifact:", artifacts.champion_model_name)

Fitting 3 folds for each of 25 candidates, totalling 75 fits
100%|██████████| 35/35 [05:32<00:00,  9.50s/trial, best loss: -0.2305349387114432] 
Notebook 06 model training completed.
Ranking champion by validation PR-AUC: xgboost_hyperopt_tuned_skew_treated
Operational champion after validation thresholding: xgboost_weighted_baseline
Saved champion model artifact: xgboost_weighted_baseline


In [ ]:
# Train-only preprocessing assurance

artifacts.preprocessing_assurance

,component,fit_scope,implementation
0,numeric_imputer,training_split_only,SimpleImputer inside Pipeline
1,categorical_imputer,training_split_only,SimpleImputer inside Pipeline
2,categorical_encoder,training_split_only,OneHotEncoder/OrdinalEncoder inside Pipeline
3,winsorization_limits,training_split_only,TrainOnlyWinsorizer inside Pipeline
4,skewness_transformer,training_split_only,Yeo-Johnson PowerTransformer inside Pipeline w...
5,numeric_scaler,training_split_only,StandardScaler inside Logistic Regression Pipe...
6,random_under_sampler,not_enabled,class weights / scale_pos_weight used by default
7,random_forest_tuning,training_split_only_cv,RandomizedSearchCV scoring=average_precision
8,xgboost_hyperopt,train_fit_validation_objective,Hyperopt fits train only and scores validation...
9,threshold_selection,validation_only,Validation threshold grid; test only confirms ...


In [ ]:
# Feature inventory and governance exclusions

display(artifacts.feature_inventory.head(80))
print("Feature inventory saved to:", TABLE_DIR / "06_feature_inventory.csv")

,feature,feature_type,model_use
0,amount,numeric,candidate_feature
1,interest_rate,numeric,candidate_feature
2,tenure_years,numeric,candidate_feature
3,total_income_pa,numeric,candidate_feature
4,dependents,numeric,candidate_feature
5,amount_missing_raw_flag,numeric,candidate_feature
6,amount_non_positive_flag,numeric,candidate_feature
7,industry_placeholder_zero_flag,numeric,candidate_feature
8,work_experience_placeholder_zero_flag,numeric,candidate_feature
9,amount_missing_flag,numeric,candidate_feature


Feature inventory saved to: D:\Banking and Finance\Projects\canadian-retail-credit-risk-xai\reports\tables\06_feature_inventory.csv


In [ ]:
# Validation results at default 0.50 threshold
# Ranking power comparison before business threshold selection

metric_cols = [
    "model_name", "dataset", "pr_auc", "roc_auc", "brier_score",
    "recall", "precision", "specificity", "f1", "balanced_accuracy",
    "mcc", "review_rate", "business_cost", "false_negative",
    "false_positive", "true_positive", "true_negative",
]

display(artifacts.validation_results[[c for c in metric_cols if c in artifacts.validation_results.columns]])

,model_name,dataset,pr_auc,roc_auc,brier_score,recall,precision,specificity,f1,balanced_accuracy,mcc,review_rate,business_cost,false_negative,false_positive,true_positive,true_negative
0,xgboost_hyperopt_tuned_skew_treated,validation,0.230535,0.752131,0.205170,0.719693,0.168920,0.648037,0.273618,0.683865,0.216698,0.385211,5782500.0,511,6455,1312,11885
1,xgboost_hyperopt_tuned,validation,0.230267,0.752052,0.205157,0.719144,0.169533,0.649836,0.274383,0.684490,0.217614,0.383524,5771000.0,512,6422,1311,11918
2,xgboost_weighted_baseline,validation,0.226251,0.751186,0.202026,0.715853,0.170923,0.654853,0.275957,0.685353,0.219168,0.378664,5755000.0,518,6330,1305,12010
3,random_forest_random_search_tuned,validation,0.224728,0.750774,0.197400,0.707076,0.174922,0.668484,0.280461,0.687780,0.223648,0.365471,5710000.0,534,6080,1289,12260
4,random_forest_weighted_baseline,validation,0.223822,0.745802,0.194638,0.684586,0.175552,0.680425,0.279445,0.682506,0.219090,0.352577,5805500.0,575,5861,1248,12479
5,logistic_regression_train_only_baseline,validation,0.220994,0.743584,0.208448,0.685134,0.172729,0.673828,0.275900,0.679481,0.214639,0.358627,5861000.0,574,5982,1249,12358


In [ ]:
# Test results at default 0.50 threshold
# Diagnostic only; not used for model/threshold selection

display(artifacts.test_results_default_threshold[[c for c in metric_cols if c in artifacts.test_results_default_threshold.columns]])

,model_name,dataset,pr_auc,roc_auc,brier_score,recall,precision,specificity,f1,balanced_accuracy,mcc,review_rate,business_cost,false_negative,false_positive,true_positive,true_negative
0,xgboost_hyperopt_tuned,test,0.216379,0.746403,0.204634,0.725727,0.170314,0.648582,0.275884,0.687155,0.220570,0.385260,5722500.0,500,6445,1323,11895
1,xgboost_hyperopt_tuned_skew_treated,test,0.216297,0.746329,0.204581,0.729018,0.170713,0.647983,0.276644,0.688500,0.222065,0.386103,5698000.0,494,6456,1329,11884
2,xgboost_weighted_baseline,test,0.214653,0.747839,0.201375,0.720790,0.173534,0.658779,0.279723,0.689784,0.224775,0.375539,5674000.0,509,6258,1314,12082
3,random_forest_random_search_tuned,test,0.212983,0.746575,0.196253,0.704334,0.175003,0.669956,0.280349,0.687145,0.223098,0.363884,5721500.0,539,6053,1284,12287
4,random_forest_weighted_baseline,test,0.211484,0.743096,0.193393,0.684586,0.176446,0.682388,0.280576,0.683487,0.220524,0.350791,5787500.0,575,5825,1248,12515
5,logistic_regression_train_only_baseline,test,0.209535,0.739734,0.207899,0.685683,0.173346,0.674973,0.276732,0.680328,0.215784,0.357635,5845500.0,573,5961,1250,12379


## Professional champion selection

The best model by PR-AUC is not automatically the final business champion.

This notebook separates:

```text
Ranking champion = strongest validation PR-AUC / ROC-AUC model.
Operational champion = best model-threshold pair under business cost and review-rate constraints.
```

This is important in credit risk because a model with slightly lower PR-AUC may still have a better operating threshold, lower review burden, or lower business cost.

In [ ]:
# All-model operational threshold comparison on validation

operational_cols = [
    "operational_rank", "model_name", "objective", "threshold",
    "pr_auc", "roc_auc", "recall", "precision", "specificity",
    "f1", "balanced_accuracy", "mcc", "review_rate", "business_cost",
    "validation_pr_auc_at_default_threshold", "validation_brier_at_default_threshold",
]

display(artifacts.all_model_threshold_shortlist[[c for c in operational_cols if c in artifacts.all_model_threshold_shortlist.columns]])

,operational_rank,model_name,objective,threshold,pr_auc,roc_auc,recall,precision,specificity,f1,balanced_accuracy,mcc,review_rate,business_cost,validation_pr_auc_at_default_threshold,validation_brier_at_default_threshold
0,1,xgboost_weighted_baseline,minimum_cost_review_rate_le_30pct,0.560,0.226251,0.751186,0.625891,0.190484,0.735605,0.292077,0.680748,0.226857,0.297079,5834500.0,0.226251,0.202026
1,2,xgboost_hyperopt_tuned,minimum_cost_review_rate_le_30pct,0.565,0.230267,0.752052,0.616566,0.189609,0.738059,0.290027,0.677312,0.223218,0.294004,5897000.0,0.230267,0.205157
2,3,xgboost_hyperopt_tuned_skew_treated,minimum_cost_review_rate_le_30pct,0.565,0.230535,0.752131,0.615469,0.189335,0.738059,0.289586,0.676764,0.222549,0.293905,5907000.0,0.230535,0.205170
3,4,random_forest_random_search_tuned,minimum_cost_review_rate_le_30pct,0.545,0.224728,0.750774,0.618760,0.187656,0.733751,0.287975,0.676256,0.220996,0.298120,5916500.0,0.224728,0.197400
4,5,random_forest_weighted_baseline,maximum_mcc,0.545,0.223822,0.745802,0.607789,0.190247,0.742857,0.289787,0.675323,0.221867,0.288846,5933000.0,0.223822,0.194638
5,6,logistic_regression_train_only_baseline,minimum_cost_review_rate_le_30pct,0.550,0.220994,0.743584,0.607789,0.185315,0.734406,0.284030,0.671098,0.214859,0.296533,6010500.0,0.220994,0.208448


In [ ]:
# Recommended operational model and threshold from validation only

display(artifacts.operational_model_threshold_recommendation)

,operational_rank,objective,model_name,dataset,threshold,roc_auc,pr_auc,brier_score,accuracy,balanced_accuracy,...,true_negative,false_positive,false_negative,true_positive,default_count,non_default_count,selection_basis,validation_pr_auc_at_default_threshold,validation_roc_auc_at_default_threshold,validation_brier_at_default_threshold
0,1,minimum_cost_review_rate_le_30pct,xgboost_weighted_baseline,validation,0.56,0.751186,0.226251,0.202026,0.725686,0.680748,...,13491,4849,682,1141,1823,18340,minimum business cost under review_rate <= 30%...,0.226251,0.751186,0.202026


In [ ]:
# Untouched test confirmation for validation-selected model/threshold

display(artifacts.test_selected_threshold_results)

,validation_operational_rank,validation_objective,model_name,dataset,threshold,roc_auc,pr_auc,brier_score,accuracy,balanced_accuracy,...,f1,mcc,review_rate,business_cost,true_negative,false_positive,false_negative,true_positive,default_count,non_default_count
0,1,minimum_cost_review_rate_le_30pct,xgboost_weighted_baseline,test,0.56,0.747839,0.214653,0.201375,0.727422,0.679973,...,0.292117,0.226423,0.294649,5848500.0,13533,4807,689,1134,1823,18340


In [ ]:
# Test confirmation for every model's validation-selected threshold
# Useful for challenger discussion

display(artifacts.test_selected_threshold_results_all_models.head(20))

,validation_operational_rank,validation_objective,model_name,dataset,threshold,roc_auc,pr_auc,brier_score,accuracy,balanced_accuracy,...,f1,mcc,review_rate,business_cost,true_negative,false_positive,false_negative,true_positive,default_count,non_default_count
0,1,minimum_cost_review_rate_le_30pct,xgboost_weighted_baseline,test,0.560,0.747839,0.214653,0.201375,0.727422,0.679973,...,0.292117,0.226423,0.294649,5848500.0,13533,4807,689,1134,1823,18340
1,2,minimum_cost_review_rate_le_30pct,xgboost_hyperopt_tuned,test,0.565,0.746403,0.216379,0.204634,0.730794,0.675899,...,0.290272,0.222584,0.288896,5922500.0,13625,4715,713,1110,1823,18340
2,3,minimum_cost_review_rate_le_30pct,xgboost_hyperopt_tuned_skew_treated,test,0.565,0.746329,0.216297,0.204581,0.731687,0.676637,...,0.291143,0.223701,0.288102,5909000.0,13642,4698,712,1111,1823,18340
3,4,minimum_cost_review_rate_le_30pct,random_forest_random_search_tuned,test,0.545,0.746575,0.212983,0.196253,0.728959,0.679584,...,0.292373,0.226390,0.292615,5855500.0,13569,4771,694,1129,1823,18340
4,5,maximum_mcc,random_forest_weighted_baseline,test,0.545,0.743096,0.211484,0.193393,0.735456,0.676486,...,0.292385,0.224606,0.283440,5911500.0,13727,4613,721,1102,1823,18340
5,6,minimum_cost_review_rate_le_30pct,logistic_regression_train_only_baseline,test,0.550,0.739734,0.209535,0.207899,0.725041,0.673231,...,0.286303,0.217898,0.294847,5971500.0,13507,4833,711,1112,1823,18340


In [ ]:
# Confusion matrices

display(artifacts.confusion_matrices_validation.head(40))
display(artifacts.confusion_matrices_test.head(40))

,model_name,dataset,threshold,actual,predicted,count,cell
0,xgboost_hyperopt_tuned_skew_treated,validation,0.5,0,0,11885,true_negative
1,xgboost_hyperopt_tuned_skew_treated,validation,0.5,0,1,6455,false_positive
2,xgboost_hyperopt_tuned_skew_treated,validation,0.5,1,0,511,false_negative
3,xgboost_hyperopt_tuned_skew_treated,validation,0.5,1,1,1312,true_positive
4,xgboost_hyperopt_tuned,validation,0.5,0,0,11918,true_negative
5,xgboost_hyperopt_tuned,validation,0.5,0,1,6422,false_positive
6,xgboost_hyperopt_tuned,validation,0.5,1,0,512,false_negative
7,xgboost_hyperopt_tuned,validation,0.5,1,1,1311,true_positive
8,xgboost_weighted_baseline,validation,0.5,0,0,12010,true_negative
9,xgboost_weighted_baseline,validation,0.5,0,1,6330,false_positive


,model_name,dataset,threshold,actual,predicted,count,cell
0,xgboost_hyperopt_tuned,test,0.5,0,0,11895,true_negative
1,xgboost_hyperopt_tuned,test,0.5,0,1,6445,false_positive
2,xgboost_hyperopt_tuned,test,0.5,1,0,500,false_negative
3,xgboost_hyperopt_tuned,test,0.5,1,1,1323,true_positive
4,xgboost_hyperopt_tuned_skew_treated,test,0.5,0,0,11884,true_negative
5,xgboost_hyperopt_tuned_skew_treated,test,0.5,0,1,6456,false_positive
6,xgboost_hyperopt_tuned_skew_treated,test,0.5,1,0,494,false_negative
7,xgboost_hyperopt_tuned_skew_treated,test,0.5,1,1,1329,true_positive
8,xgboost_weighted_baseline,test,0.5,0,0,12082,true_negative
9,xgboost_weighted_baseline,test,0.5,0,1,6258,false_positive


In [ ]:
# Lift table and calibration table on validation

display(artifacts.lift_tables_validation.head(30))
display(artifacts.calibration_tables_validation.head(30))

,model_name,risk_decile,row_count,default_count,avg_score,min_score,max_score,default_rate,lift_vs_average,cumulative_default_capture,cumulative_review_rate
0,logistic_regression_train_only_baseline,1,2017,520,0.790528,0.724702,0.942071,0.257809,2.851451,0.285244,0.100035
1,logistic_regression_train_only_baseline,2,2016,340,0.674856,0.628590,0.724610,0.168651,1.865335,0.471750,0.200020
2,logistic_regression_train_only_baseline,3,2016,253,0.588281,0.546789,0.628589,0.125496,1.388029,0.610532,0.300005
3,logistic_regression_train_only_baseline,4,2016,198,0.507020,0.470591,0.546775,0.098214,1.086283,0.719144,0.399990
4,logistic_regression_train_only_baseline,5,2017,188,0.437209,0.405661,0.470529,0.093208,1.030909,0.822271,0.500025
5,logistic_regression_train_only_baseline,6,2016,108,0.377999,0.350723,0.405617,0.053571,0.592518,0.881514,0.600010
6,logistic_regression_train_only_baseline,7,2016,93,0.324484,0.296736,0.350711,0.046131,0.510224,0.932529,0.699995
7,logistic_regression_train_only_baseline,8,2016,52,0.264228,0.229887,0.296718,0.025794,0.285287,0.961053,0.799980
8,logistic_regression_train_only_baseline,9,2016,45,0.191409,0.153184,0.229862,0.022321,0.246883,0.985738,0.899965
9,logistic_regression_train_only_baseline,10,2017,26,0.115988,0.020480,0.153142,0.012890,0.142573,1.000000,1.000000


,model_name,score_band,row_count,observed_default_rate,average_predicted_probability,calibration_gap
0,logistic_regression_train_only_baseline,"(0.999, 2017.2]",2017,0.012890,0.115988,0.103098
1,logistic_regression_train_only_baseline,"(2017.2, 4033.4]",2016,0.022321,0.191409,0.169087
2,logistic_regression_train_only_baseline,"(4033.4, 6049.6]",2016,0.025794,0.264228,0.238435
3,logistic_regression_train_only_baseline,"(6049.6, 8065.8]",2016,0.046131,0.324484,0.278353
4,logistic_regression_train_only_baseline,"(8065.8, 10082.0]",2017,0.053545,0.378013,0.324468
5,logistic_regression_train_only_baseline,"(10082.0, 12098.2]",2016,0.093254,0.437224,0.343970
6,logistic_regression_train_only_baseline,"(12098.2, 14114.4]",2016,0.098214,0.507020,0.408805
7,logistic_regression_train_only_baseline,"(14114.4, 16130.6]",2016,0.125496,0.588281,0.462785
8,logistic_regression_train_only_baseline,"(16130.6, 18146.8]",2016,0.168651,0.674856,0.506205
9,logistic_regression_train_only_baseline,"(18146.8, 20163.0]",2017,0.257809,0.790528,0.532719


In [ ]:
# Hyperparameter tuning trials

display(artifacts.tuning_trials.head(30))
print("Tuning trials saved to:", TABLE_DIR / "06_tuning_trials.csv")

,model_name,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__bootstrap,param_model__max_depth,param_model__max_features,param_model__min_samples_leaf,param_model__min_samples_split,...,learning_rate,max_depth,min_child_weight,n_estimators,reg_alpha,reg_lambda,subsample,validation_pr_auc,loss,elapsed_seconds
0,random_forest_random_search_tuned,327.036119,2.899805,13.528923,1.005127,True,10,0.7,26,393,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,random_forest_random_search_tuned,311.953902,1.898566,9.897758,4.539535,True,None,0.7,89,394,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,random_forest_random_search_tuned,276.822283,0.486194,17.690688,0.334295,True,8,0.5,45,210,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,random_forest_random_search_tuned,241.770737,5.818073,1.210068,0.274478,True,None,0.7,123,221,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,random_forest_random_search_tuned,301.960595,1.121213,13.199433,0.084292,True,14,0.7,127,171,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,random_forest_random_search_tuned,186.668724,3.424634,22.561452,0.601444,True,None,0.5,117,320,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,random_forest_random_search_tuned,189.071856,33.125519,14.647556,4.211505,True,16,0.7,113,98,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,random_forest_random_search_tuned,260.111718,0.500166,19.836325,0.515933,True,10,0.35,112,149,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,random_forest_random_search_tuned,141.466535,0.685758,12.030531,0.583694,True,16,0.35,132,104,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,random_forest_random_search_tuned,91.907163,0.982085,15.278845,0.468781,True,8,0.5,116,313,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Tuning trials saved to: D:\Banking and Finance\Projects\canadian-retail-credit-risk-xai\reports\tables\06_tuning_trials.csv


In [ ]:
# Experiment tracking log

display(artifacts.experiment_log.tail(20))
print("Local experiment tracking log saved to:", TABLE_DIR / "06_experiment_tracking_log.csv")

,experiment_id,timestamp_utc,model_name,model_family,stage,notes,param_class_weight,param_skewness_treatment,param_scaler,metric_validation_model_name,...,param_gamma,param_learning_rate,param_max_depth,param_min_child_weight,param_n_estimators,param_reg_alpha,param_reg_lambda,param_subsample,param_sampler,param_sampling_strategy
12,sampling_challenger_random_forest_under_sample...,2026-06-17T18:06:47.823687+00:00,random_forest_under_sampled_challenger,random_forest,sampling_challenger,NaN,NaN,NaN,NaN,random_forest_under_sampled_challenger,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RandomUnderSampler,0.5
13,sampling_challenger_random_forest_over_sampled...,2026-06-17T18:07:40.289463+00:00,random_forest_over_sampled_challenger,random_forest,sampling_challenger,NaN,NaN,NaN,NaN,random_forest_over_sampled_challenger,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RandomOverSampler,0.5
14,sampling_challenger_xgboost_under_sampled_chal...,2026-06-17T18:07:42.269214+00:00,xgboost_under_sampled_challenger,xgboost,sampling_challenger,NaN,NaN,NaN,NaN,xgboost_under_sampled_challenger,...,1.16156,0.035912,4.0,2.404012,250.0,0.306053,1.299173,0.856064,RandomUnderSampler,0.5
15,sampling_challenger_xgboost_over_sampled_chall...,2026-06-17T18:07:49.159482+00:00,xgboost_over_sampled_challenger,xgboost,sampling_challenger,NaN,NaN,NaN,NaN,xgboost_over_sampled_challenger,...,1.16156,0.035912,4.0,2.404012,250.0,0.306053,1.299173,0.856064,RandomOverSampler,0.5
16,baseline_logistic_regression_train_only_baseline,2026-06-17T18:45:26.999885+00:00,logistic_regression_train_only_baseline,logistic_regression,baseline,NaN,balanced,yeo_johnson,standard,logistic_regression_train_only_baseline,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,baseline_random_forest_weighted_baseline,2026-06-17T18:45:34.797590+00:00,random_forest_weighted_baseline,random_forest,baseline,NaN,balanced_subsample,winsorize_only,NaN,random_forest_weighted_baseline,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,baseline_xgboost_weighted_baseline,2026-06-17T18:45:40.559189+00:00,xgboost_weighted_baseline,xgboost,baseline,NaN,NaN,winsorize_only,NaN,xgboost_weighted_baseline,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,tuned_random_forest_random_search_tuned,2026-06-17T19:01:54.416838+00:00,random_forest_random_search_tuned,random_forest,tuned,NaN,NaN,NaN,NaN,random_forest_random_search_tuned,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,tuned_xgboost_hyperopt_tuned,2026-06-17T19:06:19.656992+00:00,xgboost_hyperopt_tuned,xgboost,tuned,NaN,NaN,NaN,NaN,xgboost_hyperopt_tuned,...,1.16156,0.035912,4.0,2.404012,250.0,0.306053,1.299173,0.856064,NaN,NaN
21,tuned_skew_treatment_variant_xgboost_hyperopt_...,2026-06-17T19:11:40.391599+00:00,xgboost_hyperopt_tuned_skew_treated,xgboost,tuned_skew_treatment_variant,NaN,NaN,yeo_johnson_train_only,NaN,xgboost_hyperopt_tuned_skew_treated,...,1.16156,0.035912,4.0,2.404012,250.0,0.306053,1.299173,0.856064,NaN,NaN


Local experiment tracking log saved to: D:\Banking and Finance\Projects\canadian-retail-credit-risk-xai\reports\tables\06_experiment_tracking_log.csv


In [ ]:
# Model readiness gate

display(artifacts.model_readiness_gate)

,check,status,evidence
0,required_splits_present,pass,train/validation/test split from Notebook 05 w...
1,train_only_preprocessing_documented,pass,imputers/encoders/scalers/winsorizer/skew tran...
2,xgboost_required_and_fitted,pass,"XGBoost is a core model, not optional"
3,random_forest_fitted,pass,Random Forest baseline and tuned challenger fi...
4,hyperopt_completed,pass,XGBoost Hyperopt tuning completed
5,rf_random_search_completed,pass,Random Forest randomized search completed
6,all_model_threshold_comparison_completed,pass,All models compared under validation-selected ...
7,test_used_only_for_confirmation,pass,Test metrics generated after model/threshold s...
8,experiment_tracking_saved,pass,Local CSV experiment log saved; Neptune optional
9,champion_model_saved,pass,Operational champion model artifact saved


In [ ]:
# Output manifest

expected_outputs = [
    "06_model_validation_results_default_threshold.csv",
    "06_model_test_results_default_threshold.csv",
    "06_model_selection_summary.csv",
    "06_validation_predictions_default_threshold.csv",
    "06_test_predictions_default_threshold.csv",
    "06_validation_threshold_grid_all_models.csv",
    "06_all_model_operational_threshold_comparison_validation.csv",
    "06_operational_model_threshold_recommendation.csv",
    "06_test_selected_threshold_results.csv",
    "06_test_selected_threshold_results_all_models.csv",
    "06_tuning_trials.csv",
    "06_confusion_matrices_validation.csv",
    "06_confusion_matrices_test.csv",
    "06_lift_tables_validation.csv",
    "06_calibration_tables_validation.csv",
    "06_preprocessing_train_only_assurance.csv",
    "06_feature_inventory.csv",
    "06_model_readiness_gate.csv",
    "06_neptune_config_status.csv",
    "06_business_cost_assumptions.csv",
]

manifest = pd.DataFrame({
    "output_file": expected_outputs,
    "exists": [(TABLE_DIR / f).exists() for f in expected_outputs],
    "path": [str(TABLE_DIR / f) for f in expected_outputs],
})
display(manifest)

model_manifest = pd.DataFrame({
    "artifact": ["06_candidate_models.joblib", "06_champion_model.joblib", "06_model_feature_metadata.joblib"],
    "exists": [(MODEL_ARTIFACT_DIR / f).exists() for f in ["06_candidate_models.joblib", "06_champion_model.joblib", "06_model_feature_metadata.joblib"]],
    "path": [str(MODEL_ARTIFACT_DIR / f) for f in ["06_candidate_models.joblib", "06_champion_model.joblib", "06_model_feature_metadata.joblib"]],
})
display(model_manifest)

,output_file,exists,path
0,06_model_validation_results_default_threshold.csv,True,D:\Banking and Finance\Projects\canadian-retai...
1,06_model_test_results_default_threshold.csv,True,D:\Banking and Finance\Projects\canadian-retai...
2,06_model_selection_summary.csv,True,D:\Banking and Finance\Projects\canadian-retai...
3,06_validation_predictions_default_threshold.csv,True,D:\Banking and Finance\Projects\canadian-retai...
4,06_test_predictions_default_threshold.csv,True,D:\Banking and Finance\Projects\canadian-retai...
5,06_validation_threshold_grid_all_models.csv,True,D:\Banking and Finance\Projects\canadian-retai...
6,06_all_model_operational_threshold_comparison_...,True,D:\Banking and Finance\Projects\canadian-retai...
7,06_operational_model_threshold_recommendation.csv,True,D:\Banking and Finance\Projects\canadian-retai...
8,06_test_selected_threshold_results.csv,True,D:\Banking and Finance\Projects\canadian-retai...
9,06_test_selected_threshold_results_all_models.csv,True,D:\Banking and Finance\Projects\canadian-retai...


,artifact,exists,path
0,06_candidate_models.joblib,True,D:\Banking and Finance\Projects\canadian-retai...
1,06_champion_model.joblib,True,D:\Banking and Finance\Projects\canadian-retai...
2,06_model_feature_metadata.joblib,True,D:\Banking and Finance\Projects\canadian-retai...


## Notebook 06 final conclusion

Carry these outputs into the next phase:

1. The operational champion model artifact: `06_champion_model.joblib`.
2. The full candidate model artifact: `06_candidate_models.joblib`.
3. The validation-selected operating threshold: `06_operational_model_threshold_recommendation.csv`.
4. The final test confirmation: `06_test_selected_threshold_results.csv`.
5. The prediction files for threshold, lift, calibration, and explainability work.

Notebook 07 should focus on stakeholder threshold policy, business-cost framing, and operational decision design. Notebook 08 should focus on SHAP, local explanations, anchors, and counterfactual diagnostics.